# Variant 1 — TaskTransformer (complete) + TimeTransformer
Pipeline: token combinato (regione+task) → predice il prossimo token, poi il tempo.

In [1]:
import os
import torch
import pandas as pd
from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments
from pm4py.objects.log.obj import Trace, Event, EventLog

from core import TaskTransformer, TimeTransformer
from core.training import train_task, train_time
from utils import get_decoding, get_encoding, hamming_distance, edit_distance_weighted_levenshtein

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
info = torch.load('../data/prepared_data.pt', map_location=device, weights_only=False)
n = info['n']

data_task = {
    'train_complete': info['data_complete'][:n],
    'val_complete': info['data_complete'][n:],
}
data_time = {
    'train_complete': info['data_complete'][:n],
    'val_complete': info['data_complete'][n:],
    'train_times': info['data_times'][:n],
    'val_times': info['data_times'][n:],
}

vocab_size = info['vocab_size_complete']
num_regions = info['num_regions']
num_tasks = info['num_tasks']

decode_complete = lambda b: [info['id_to_bit_complete'][x] for x in b]
encode_complete = lambda a: [info['bit_to_id_complete'][tuple(x)] for x in a]

net = info['net']

print(f'vocab={vocab_size}, n_train={n}')

vocab=9, n_train=3200


In [3]:
param_task_region_transformer_default = dict(block_size=256, n_embd=256, n_head=8, n_layer=3, dropout=0.30, lr=3e-4, batch_size=16, max_iters=500, eval_iters=200, eval_interval=150)
param_time_transformer_default = dict(block_size=64,  n_embd=128, n_head=8, n_layer=3, dropout=0.30, lr=3e-4, batch_size=16, max_iters=500, eval_iters=200, eval_interval=150)

params = torch.load('../data/v1_best_params.pt', map_location=device, weights_only=False) if os.path.exists('../data/v1_best_params.pt') else None

p_task_region = param_task_region_transformer_default
p_time = param_time_transformer_default
if params is not None:
    p_task_region = {**params['TaskTransformer'], 'max_iters': 3000, 'eval_iters': 500, 'eval_interval': 500}
    p_time = {**params['TimeTransformer'],  'max_iters': 3000, 'eval_iters': 500, 'eval_interval': 500}

print('Task Region params:', p_task_region)
print('Time params:', p_time)

Task Region params: {'block_size': 256, 'n_embd': 256, 'n_head': 8, 'n_layer': 3, 'dropout': 0.3, 'lr': 0.0003, 'batch_size': 16, 'max_iters': 500, 'eval_iters': 200, 'eval_interval': 150}
Time params: {'block_size': 64, 'n_embd': 128, 'n_head': 8, 'n_layer': 3, 'dropout': 0.3, 'lr': 0.0003, 'batch_size': 16, 'max_iters': 500, 'eval_iters': 200, 'eval_interval': 150}


In [4]:
model_task = TaskTransformer(
    task_vocab_size=vocab_size,
    block_size=p_task_region['block_size'],
    n_embd=p_task_region['n_embd'],
    dropout=p_task_region['dropout'],
    n_head=p_task_region['n_head'],
    n_layer=p_task_region['n_layer'],
).to(device)

train_task(model_task, data_task, p_task_region, device, data_key='complete', printing=True)
print('TaskTransformer trained.')

step 0: train loss 2.0280, val loss 2.0320
step 150: train loss 0.5658, val loss 0.5602
step 300: train loss 0.5592, val loss 0.5566
step 450: train loss 0.5553, val loss 0.5523
step 499: train loss 0.5549, val loss 0.5535
TaskTransformer trained.


In [5]:
model_time = TimeTransformer(
    vocab_size_task=vocab_size,
    block_size=p_time['block_size'],
    n_embd=p_time['n_embd'],
    dropout=p_time['dropout'],
    n_head=p_time['n_head'],
    n_layer=p_time['n_layer'],
    separated_regions=False,
).to(device)

train_time(model_time, data_time, p_time, device, printing=True)
print('TimeTransformer trained.')

step 0: train loss 0.2477, val loss 0.2493
step 150: train loss 0.0322, val loss 0.0331
step 300: train loss 0.0100, val loss 0.0115
step 450: train loss 0.0061, val loss 0.0066
step 499: train loss 0.0041, val loss 0.0047
TimeTransformer trained.


In [6]:
max_new_tokens = 100

sep_id = info['bit_to_id_complete'][tuple([0]*(num_regions+num_tasks))]
mean_sep_delta = info['data_times'][:n][info['data_complete'][:n] == sep_id].float().mean().item()
print(mean_sep_delta)

context = torch.tensor(encode_complete([[0]*(num_regions+num_tasks)]), dtype=torch.long, device=device).unsqueeze(0)
context_time = torch.tensor([mean_sep_delta], dtype=torch.float32, device=device).unsqueeze(0)

generated_ids = []
generated_times = []

for _ in range(max_new_tokens):
    next_id = model_task.predict_next_task(idx_task=context, block_size=p_task_region['block_size'])
    context = torch.cat((context, next_id), dim=1)
    context_aligned = context[:, 1:]   # allineamento offset

    next_time = model_time.predict_next_time(idx_tasks=context_aligned, idx_times=context_time, block_size=p_time['block_size'])
    context_time = torch.cat((context_time, next_time), dim=1)

    generated_ids.append(next_id.item())
    generated_times.append(next_time.item())

decoded = decode_complete(generated_ids)
'''for i, (step, t) in enumerate(zip(decoded, generated_times)):
    print(f'{i:02d}: {[int(b) for b in step]} — time={round(t,4)}')'''

0.5568078756332397


"for i, (step, t) in enumerate(zip(decoded, generated_times)):\n    print(f'{i:02d}: {[int(b) for b in step]} — time={round(t,4)}')"

In [7]:
# Raggruppa la sequenza in tracce separate (separatore = vettore zero)
traces_generated, current = [], []
for step in decoded:
    bits = [int(b) for b in step]
    current.append(bits)
    if bits == [0]*(num_regions+num_tasks):
        if len(current) > 1:
            traces_generated.append(current)
        current = []

for i,trace in enumerate(traces_generated):
    print(f"{i}: {trace}")

0: [[1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
1: [[1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
2: [[1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1], [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
3: [[1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1], [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
4: [[1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
5: [[1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1], [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
6: [[1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1], [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0,

In [8]:
traces_decoded = get_decoding(traces_generated, net.regions, net.tasks)
print(traces_decoded)

[['start_T1', 'end_T1', 'start_T2', 'end_T2'], ['start_T1', 'end_T1', 'start_T3', 'end_T3'], ['start_T5', 'start_T4', 'end_T4', 'end_T5'], ['start_T5', 'start_T4', 'end_T4', 'end_T5'], ['start_T1', 'end_T1', 'start_T2', 'end_T2'], ['start_T5', 'start_T4', 'end_T5', 'end_T4'], ['start_T5', 'start_T4', 'end_T5', 'end_T4'], ['start_T5', 'start_T4', 'end_T5', 'end_T4'], ['start_T1', 'end_T1', 'start_T3', 'end_T3'], ['start_T5', 'start_T4', 'end_T5', 'end_T4'], ['start_T1', 'end_T1', 'start_T2', 'end_T2'], ['start_T1', 'end_T1', 'start_T3', 'end_T3'], ['start_T4', 'start_T5', 'end_T4', 'end_T5'], ['start_T5', 'start_T4', 'end_T4', 'end_T5'], ['start_T5', 'start_T4', 'end_T5', 'end_T4'], ['start_T5', 'end_T5', 'start_T4', 'end_T4'], ['start_T5', 'start_T4', 'end_T4', 'end_T5'], ['start_T5', 'end_T5', 'start_T4', 'end_T4'], ['start_T1', 'end_T1', 'start_T2', 'end_T2'], ['start_T4', 'start_T5', 'end_T5', 'end_T4'], ['start_T4', 'start_T5', 'end_T4', 'end_T5'], ['start_T5', 'start_T4', 'end_T5'

In [9]:
'''
PROBLEMA: Può generare tracce sfasate, con più eventi per step.
get_decoding non funziona sotto questo punto di vista. o meglio, funziona generando eventi in più (tipo 6 eventi da 4 step perchè ci sono 2 step generati male)
'''

classifier_dict_tasks = info['classifier_dict_tasks']
dict_task_step_encoding = info['dict_task_step_encoding']
current_trace_context = []
#traces_decoded_list = [step for trace in traces_decoded for step in trace]

tasks_previous = [0] * num_tasks
for i, (step, t) in enumerate(zip(decoded, generated_times)):
    bits = [int(b) for b in step]
    is_sep = bits == [0]*(num_regions+num_tasks)
    tasks_step = bits[num_regions:]

    step_events = []
    for j, task in enumerate(tasks_step):
        if task != tasks_previous[j]:
            step_events.append(("start_" if task == 1 else "end_") + net.tasks[j])
    tasks_previous = tasks_step

    if len(step_events) == 1:
        event_name = step_events[0]
        if event_name in classifier_dict_tasks:
            rt, max_len = classifier_dict_tasks[event_name]
            ctx = list(reversed(current_trace_context))[:max_len]
            padded = ctx + ['PAD'] * (max_len - len(ctx))
            encoded = [dict_task_step_encoding.get(s, dict_task_step_encoding['PAD']) for s in padded]
            expected = round(rt.predict([encoded])[0], 4)
        else: # Non ci dovrebbe mai entrare in teoria
            expected = 0.0 if not current_trace_context else "n/d"
        current_trace_context.append(event_name)
        note = ""
    elif len(step_events) == 0: # step che non genera eventi (es. cambia solo la regione)
        expected, note = "—", "(step senza evento)"
    else:# step malformato: accende/spegne 2 task insieme
        expected, note = "—", f"(step ambiguo: {step_events})"

    if is_sep:
        current_trace_context = []

    print(f'{i:02d}: {bits} — time={round(t,4)} | expected={expected} {note}')

00: [1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0] — time=0.051 | expected=0.0 
01: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=0.9603 | expected=1.0 
02: [1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0] — time=0.9992 | expected=1.0 
03: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=0.9916 | expected=1.0 
04: [1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0] — time=0.0324 | expected=0.0 
05: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=0.9178 | expected=1.0 
06: [1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0] — time=0.9791 | expected=1.0 
07: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=0.9805 | expected=1.0 
08: [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1] — time=0.0309 | expected=0.0 
09: [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1] — time=0.4114 | expected=0.4286 
10: [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1] — time=0.3393 | expected=0.4286 
11: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=0.2001 | expected=0.1429 
12: [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1] — time=0.0598 | expected=0.0 
13: [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1] — time=0.4316 | expected=0.4286 
14: [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1]

In [10]:
# Allineamento con conformance checking pm4py
check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
loop = tuple(["back_L"]) + tuple(["end_L"])
silent_prefixes = start + end + loop

# Creo i parametri per l'allineamento
model_cost, sync_cost = {}, {}
for t in net.net.transitions: # Prendo tutte le transizioni
    if t.label is None or (t.label is not None and t.label.startswith(silent_prefixes)): # Se è una transizione silente
        model_cost[t] = 0
        sync_cost[t] = 10000
    else: # Se è un task vero e proprio
        model_cost[t] = 10000
        sync_cost[t] = 0

alignment_params = {
    alignments.Parameters.PARAM_MODEL_COST_FUNCTION: model_cost,
    alignments.Parameters.PARAM_SYNC_COST_FUNCTION: sync_cost,
}

# Creiamo l'EventLog di ogni traccia per poi poterla allineare
eventlog_traces = EventLog()
for trace in traces_decoded:
    t = Trace()
    for activity in trace:
        t.append(Event({'concept:name': activity}))
    eventlog_traces.append(t)

aligned_traces = alignments.apply(eventlog_traces, net.net, net.initial_marking, net.final_marking, parameters=alignment_params)
for i,trace in enumerate(aligned_traces):
    print(f"{i}: {trace}")

aligning log, completed variants ::   0%|          | 0/7 [00:00<?, ?it/s]

0: {'alignment': [('>>', 'start_X0'), ('>>', 'start_L2'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('>>', 'end_L2'), ('>>', 'start_X3'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('>>', 'end_X3'), ('>>', 'end_X0')], 'cost': 0, 'visited_states': 10, 'queued_states': 25, 'traversed_arcs': 25, 'lp_solved': 1, 'fitness': 1.0, 'bwc': 80000}
1: {'alignment': [('>>', 'start_X0'), ('>>', 'start_L2'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('>>', 'end_L2'), ('>>', 'start_X3'), ('start_T3', 'start_T3'), ('end_T3', 'end_T3'), ('>>', 'end_X3'), ('>>', 'end_X0')], 'cost': 0, 'visited_states': 10, 'queued_states': 25, 'traversed_arcs': 25, 'lp_solved': 1, 'fitness': 1.0, 'bwc': 80000}
2: {'alignment': [('>>', 'start_X0'), ('>>', 'start_P4'), ('start_T5', 'start_T5'), ('>>', 'start_L5'), ('start_T4', 'start_T4'), ('end_T4', 'end_T4'), ('end_T5', 'end_T5'), ('>>', 'end_L5'), ('>>', 'end_P4'), ('>>', 'end_X0')], 'cost': 0, 'visited_states': 12, 'queued_states': 36, 'traversed_arcs

In [11]:
'''Codifichiamo le tracce allineate (per poi poterle confrontare con quelle generate dal transformer)'''

silent_prefixes = start + end + tuple(["back_L"])

aligned_traceEncoded_regions, aligned_traceEncoded_tasks = get_encoding(
    [[step for _, step in a['alignment'] if step and not step.startswith(silent_prefixes) and step != '>>']
     for a in aligned_traces],
    net.regions, net.tasks, net.open_clauses, net.end_clauses
)

print(aligned_traceEncoded_regions)

df_aligned_traces = pd.concat([aligned_traceEncoded_regions, aligned_traceEncoded_tasks], axis=0)

df_aligned_traces

    0   1   2   3   4   5   6   7   8   9   ...  90  91  92  93  94  95  96  \
R0   1   1   1   0   1   1   1   0   1   1  ...   1   0   1   1   1   0   1   
R1   1   1   1   0   1   1   1   0   0   0  ...   0   0   0   0   0   0   1   
R2   1   0   0   0   1   0   0   0   0   0  ...   0   0   0   0   0   0   1   
R3   0   0   1   0   0   0   1   0   0   0  ...   0   0   0   0   0   0   0   
R4   0   0   0   0   0   0   0   0   1   1  ...   1   0   1   1   1   0   0   
R5   0   0   0   0   0   0   0   0   0   1  ...   1   0   0   1   1   0   0   

    97  98  99  
R0   1   1   0  
R1   1   1   0  
R2   0   0   0  
R3   0   1   0  
R4   0   0   0  
R5   0   0   0  

[6 rows x 100 columns]


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
R0,1,1,1,0,1,1,1,0,1,1,...,1,0,1,1,1,0,1,1,1,0
R1,1,1,1,0,1,1,1,0,0,0,...,0,0,0,0,0,0,1,1,1,0
R2,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
R3,0,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
R4,0,0,0,0,0,0,0,0,1,1,...,1,0,1,1,1,0,0,0,0,0
R5,0,0,0,0,0,0,0,0,0,1,...,1,0,0,1,1,0,0,0,0,0
T1,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
T2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
T3,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
T4,0,0,0,0,0,0,0,0,0,1,...,1,0,0,1,0,0,0,0,0,0


In [12]:
'''Creo una lista delle tracce codificate (ogni traccia è una lista dove ogni elemento è una colonna del df, ossia uno step) --> più facili da confrontare quando calcoliamo la distanza'''

aligned_traces_encoded = []
aligned_trace_encoded = []
for element in df_aligned_traces.T.values:
    element = element.tolist()
    aligned_trace_encoded.append(element)
    if element == [0] * (num_regions+num_tasks):
        aligned_traces_encoded.append(aligned_trace_encoded)
        aligned_trace_encoded = []

# Andiamo a calcolare il costo con la edit distance (weighted_levenshtein)
costs = []
for i, (gen, aln) in enumerate(zip(traces_generated, aligned_traces_encoded)):
    cost = edit_distance_weighted_levenshtein(gen, aln, num_regions+num_tasks, num_regions+num_tasks, hamming_distance)
    costs.append(cost)
    print(f'Traccia {i}: edit_distance={cost}')

print(f'\nEdit distance media: {sum(costs)/len(costs):.2f}')

Traccia 0: edit_distance=0.0
Traccia 1: edit_distance=0.0
Traccia 2: edit_distance=1.0
Traccia 3: edit_distance=1.0
Traccia 4: edit_distance=0.0
Traccia 5: edit_distance=0.0
Traccia 6: edit_distance=0.0
Traccia 7: edit_distance=0.0
Traccia 8: edit_distance=0.0
Traccia 9: edit_distance=0.0
Traccia 10: edit_distance=0.0
Traccia 11: edit_distance=0.0
Traccia 12: edit_distance=1.0
Traccia 13: edit_distance=1.0
Traccia 14: edit_distance=0.0
Traccia 15: edit_distance=0.0
Traccia 16: edit_distance=1.0
Traccia 17: edit_distance=0.0
Traccia 18: edit_distance=0.0
Traccia 19: edit_distance=0.0
Traccia 20: edit_distance=1.0
Traccia 21: edit_distance=0.0
Traccia 22: edit_distance=0.0
Traccia 23: edit_distance=1.0
Traccia 24: edit_distance=0.0

Edit distance media: 0.28
